# Retrieval Explained: Data, Pipeline, and Model Behavior

This notebook explains:
- what the competition data looks like,
- how retrieval inputs are prepared,
- why lexical models rank documents the way they do,
- and how we evaluate retrieval quality.


## 1. Load Data

We load document corpus, train queries, and relevance judgments (`qgts_train.json`).


In [ ]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path('../../data')

docs_df = pd.read_json(DATA_DIR / 'docs.json')
train_queries_df = pd.read_json(DATA_DIR / 'queries_train.json')

with open(DATA_DIR / 'qgts_train.json', 'r', encoding='utf-8') as f:
    qrels_raw = json.load(f)

print('docs:', docs_df.shape)
print('train queries:', train_queries_df.shape)
print('qrels entries:', len(qrels_raw))


## 2. Understand the Data Fields

- `docs`: each row is a candidate document with `id`, `title`, `text`, `tags`, `category`.
- `queries_train`: each row is a query with `id`, `title`, `text`, etc.
- `qrels`: for each query ID, a list of relevant document IDs.


In [ ]:
display(docs_df.head(2))
display(train_queries_df.head(2))

first_qid = next(iter(qrels_raw))
print('example query id:', first_qid)
print('example qrels:', qrels_raw[first_qid])


## 3. Build the Retrieval Input (`content`)

Retrieval models expect one text per item. We build:
- document content: `title + text + tags`
- query content: `title + text`

This makes all models consume exactly the same textual signal.


In [ ]:
def value_to_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def create_content_column(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''

    content = []
    for _, row in out[columns].iterrows():
        merged = ' '.join(value_to_text(row[col]) for col in columns).strip().lower()
        content.append(merged)

    out['content'] = content
    out['id'] = out['id'].astype(str)
    return out


docs_proc = create_content_column(docs_df, ['title', 'text', 'tags'])
train_proc = create_content_column(train_queries_df, ['title', 'text'])

docs_proc[['id', 'content']].head(2)


## 4. Why TF-IDF and BM25 Work Differently

Both are lexical methods, but they score terms differently:
- TF-IDF: weights query-document overlap by term rarity.
- BM25: adds saturation and document-length normalization.

BM25 often handles long documents better because term frequency gains are capped.


In [ ]:
import numpy as np
import re
from rank_bm25 import BM25Plus
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

token_pattern = re.compile(r'[a-z0-9]+')

def tokenize(text):
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return token_pattern.findall(txt)


# Use a small slice so the notebook runs quickly.
docs_small = docs_proc.head(5000).copy()
queries_small = train_proc.head(100).copy()

# TF-IDF scores
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
try:
    doc_vectors = vectorizer.fit_transform(docs_small['content'])
except ValueError as error:
    if 'After pruning, no terms remain' not in str(error):
        raise
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
    doc_vectors = vectorizer.fit_transform(docs_small['content'])

query_vectors = vectorizer.transform(queries_small['content'])
tfidf_scores = cosine_similarity(query_vectors, doc_vectors)

# BM25 scores
tokenized_corpus = [tokenize(text) for text in docs_small['content']]
bm25 = BM25Plus(tokenized_corpus)

q0_tokens = tokenize(queries_small.iloc[0]['content'])
bm25_q0 = bm25.get_scores(q0_tokens)

print('TF-IDF score matrix shape:', tfidf_scores.shape)
print('BM25 first-query score vector shape:', bm25_q0.shape)


## 5. Retrieval Evaluation Metrics

For each query with ground truth:
- Precision@K: fraction of top-K docs that are relevant.
- Recall@K: fraction of relevant docs found in top-K.
- MRR@K: inverse rank of first relevant result.
- MAP@K: average precision over the ranked list, then mean across queries.


In [ ]:
def build_qrels_lookup(qrels_raw):
    out = {}
    for qid, info in qrels_raw.items():
        out[str(qid)] = [str(item['doc_id']) for item in info.get('relevant_doc_ids', [])]
    return out


def average_precision_at_k(retrieved, relevant_set, k):
    if not relevant_set:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for rank, doc_id in enumerate(retrieved[:k], start=1):
        if doc_id in relevant_set:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / len(relevant_set)


def evaluate(results, qrels, k=10):
    recalls, precisions, mrrs, maps = [], [], [], []

    for item in results:
        qid = str(item['query_id'])
        retrieved = [str(doc) for doc in item['relevant_docs'][:k]]
        relevant = set(qrels.get(qid, []))
        if not relevant:
            continue

        hits = sum(1 for doc in retrieved if doc in relevant)
        recalls.append(hits / len(relevant))
        precisions.append(hits / max(1, len(retrieved)))

        rr = 0.0
        for rank, doc_id in enumerate(retrieved, start=1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrrs.append(rr)

        maps.append(average_precision_at_k(retrieved, relevant, k))

    return {
        'avg_precision': float(np.mean(precisions)) if precisions else float('nan'),
        'avg_recall': float(np.mean(recalls)) if recalls else float('nan'),
        'mrr': float(np.mean(mrrs)) if mrrs else float('nan'),
        'map': float(np.mean(maps)) if maps else float('nan'),
    }


# Quick TF-IDF retrieval on the small slice.
doc_ids_small = docs_small['id'].astype(str).to_numpy()
tfidf_results = []
for i, row_scores in enumerate(tfidf_scores):
    top_idx = np.argsort(row_scores)[-10:][::-1]
    tfidf_results.append({
        'query_id': queries_small.iloc[i]['id'],
        'relevant_docs': doc_ids_small[top_idx].tolist(),
    })

# Use qrels only for the same query subset.
qrels_lookup = build_qrels_lookup(qrels_raw)
qrels_subset = {qid: qrels_lookup.get(qid, []) for qid in queries_small['id'].astype(str)}

metrics = evaluate(tfidf_results, qrels_subset, k=10)
metrics


## 6. Why This Pipeline Is Structured This Way

- One shared preprocessing stage keeps model comparisons fair.
- Lexical retrieval (TF-IDF/BM25) is strong and fast for this type of StackOverflow-like text data.
- Evaluation on train queries prevents blind tuning on leaderboard scores.
- Kaggle notebook is kept minimal so score reproduction is straightforward.


## Phase-1 Comparison Notebook

For direct Phase-1 benchmarking across **TF-IDF**, **BM25+**, and **Embedding-based retrieval**, use:
- `notebooks/phase1/phase1_retrieval_basics.ipynb`

That notebook runs metric-based comparison tables with Precision@K, Recall@K, MRR@K, and MAP@K.
